# 📊 Data Analysis — 10-Week Course
## Week 10: Capstone Project

---
### Learning Objectives
By the end of this week, you will be able to:
- Execute a complete, professional-grade data analysis pipeline from scratch
- Produce a structured report with visualisations and statistical evidence
- Communicate findings clearly to a non-technical audience
- Reflect critically on methodological choices and limitations

### Overview
The capstone integrates **all 10 weeks** into a single end-to-end project on the Africa Health Dataset:

```
Part 1  — Data Acquisition & Quality Assessment
Part 2  — Data Cleaning & Feature Engineering
Part 3  — Exploratory Data Analysis
Part 4  — Statistical Testing
Part 5  — Modelling (Regression + Classification)
Part 6  — Visualisation Dashboard
Part 7  — Written Report & Recommendations
```

**Grading:** 100 points total (see breakdown at the end)

---
## Course Reference: Skills Learned

| Week | Topic | Key Skills |
|---|---|---|
| 1 | Introduction | Data types, workflow, environment |
| 2 | Data Collection | CSV, Excel, JSON, SQL, APIs |
| 3 | Data Cleaning | Missing values, outliers, normalisation |
| 4 | EDA | Descriptive stats, histograms, correlations |
| 5 | Visualisation | Design principles, Matplotlib, Seaborn, GridSpec |
| 6 | Statistics | t-tests, chi-square, ANOVA, CIs, effect size |
| 7 | Regression | Linear, multiple, logistic, diagnostics |
| 8 | Classification | Decision trees, kNN, k-means, evaluation |
| 9 | Advanced | Feature engineering, PCA, SQL, regularisation |
| **10** | **Capstone** | **Full end-to-end pipeline** |

In [ ]:
# ── Capstone Setup: All Imports ───────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, ttest_ind, f_oneway, pearsonr, spearmanr
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    r2_score, mean_squared_error, accuracy_score,
    classification_report, confusion_matrix,
    roc_curve, auc, silhouette_score
)
import sqlite3, os, json, warnings
from datetime import datetime
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi"      : 110,
    "axes.spines.top" : False,
    "axes.spines.right": False,
    "axes.grid"       : True,
    "grid.alpha"      : 0.25,
    "font.size"       : 10,
})

PALETTE = {
    "West Africa"    : "#3498DB",
    "East Africa"    : "#2ECC71",
    "North Africa"   : "#E74C3C",
    "Central Africa" : "#9B59B6",
    "Southern Africa": "#F39C12",
}

os.makedirs("capstone_output", exist_ok=True)
print("Setup complete. Output folder: capstone_output/")

---
## Part 1 — Data Acquisition & Quality Assessment (15 pts)

In [ ]:
# ── Generate the raw dataset (with deliberate issues) ─────────────────
np.random.seed(42)
N = 54
countries = [
    "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
    "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
    "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
    "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
    "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
    "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
    "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
    "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
]
regions = (["West Africa"]*16 + ["East Africa"]*14 +
           ["North Africa"]*6  + ["Central Africa"]*8 + ["Southern Africa"]*10)

raw = pd.DataFrame({
    "country"           : countries,
    "region"            : regions,
    "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
    "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
    "maternal_mortality": np.clip(np.random.exponential(350, N), 20, 2000),
    "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
    "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
    "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
    "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
    "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
})

# Introduce some quality issues
np.random.seed(99)
missing_idx = np.random.choice(N, size=8, replace=False)
for col in ["health_expenditure", "vaccination_dtp3", "water_access"]:
    raw.loc[np.random.choice(missing_idx, size=3, replace=False), col] = np.nan
raw = pd.concat([raw, raw.iloc[[5, 20]]], ignore_index=True)  # duplicates

raw.to_csv("capstone_output/raw_data.csv", index=False)
print(f"Raw dataset: {raw.shape[0]} rows × {raw.shape[1]} columns")
print(f"Missing values per column:")
print(raw.isnull().sum()[raw.isnull().sum() > 0].to_string())
print(f"Duplicate rows: {raw.duplicated().sum()}")

# Quality dimensions
num_cols = raw.select_dtypes("number").columns
print("\nData Quality Summary:")
print(f"  Completeness : {1 - raw.isnull().mean().mean():.1%}")
print(f"  Uniqueness   : {1 - raw.duplicated().mean():.1%}")
print(f"  Numeric cols : {len(num_cols)}")

---
## Part 2 — Data Cleaning & Feature Engineering (15 pts)

In [ ]:
# ── Cleaning pipeline ─────────────────────────────────────────────────
df = raw.copy()

# 1. Remove duplicates
before_dedup = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Duplicates removed: {before_dedup - len(df)}")

# 2. Impute missing values with region-conditional median
impute_cols = ["health_expenditure", "vaccination_dtp3", "water_access"]
for col in impute_cols:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        regional_median = df.groupby("region")[col].transform("median")
        global_median   = df[col].median()
        df[col] = df[col].fillna(regional_median).fillna(global_median)
        print(f"  Imputed {n_missing} missing values in {col}")

print(f"Missing after imputation: {df.isnull().sum().sum()}")

# ── Feature engineering ───────────────────────────────────────────────
# Log transforms
df["log_gdp"]           = np.log(df["gdp_per_capita"])
df["log_infant_mort"]   = np.log(df["infant_mortality"])
df["log_maternal_mort"] = np.log(df["maternal_mortality"])

# Composite Health Index (0–100)
mm = MinMaxScaler()
le_n   = mm.fit_transform(df[["life_expectancy"]])
im_n   = 1 - mm.fit_transform(df[["infant_mortality"]])
vax_n  = mm.fit_transform(df[["vaccination_dtp3"]])
wat_n  = mm.fit_transform(df[["water_access"]])
df["health_index"] = (0.35*le_n + 0.25*im_n + 0.20*vax_n + 0.20*wat_n).flatten() * 100

# Composite Disease Burden (0–100, higher = worse)
hiv_n = mm.fit_transform(df[["hiv_prevalence"]])
mm_n  = mm.fit_transform(df[["maternal_mortality"]])
df["disease_burden"] = (0.4*im_n[::-1] + 0.35*hiv_n + 0.25*mm_n).flatten() * 100  # im inverted back
df["disease_burden"] = (0.4*(1-im_n) + 0.35*hiv_n + 0.25*mm_n).flatten() * 100

# Binary targets
df["high_le"]  = (df["life_expectancy"] >= df["life_expectancy"].median()).astype(int)
df["gdp_band"] = pd.qcut(df["gdp_per_capita"], q=3, labels=["Low","Mid","High"])

df.to_csv("capstone_output/clean_data.csv", index=False)
print(f"\nClean dataset saved: {df.shape}")
print("New features:", [c for c in df.columns if c not in raw.columns])

---
## Part 3 — Exploratory Data Analysis (15 pts)

In [ ]:
num_cols = ["life_expectancy","infant_mortality","maternal_mortality",
            "hiv_prevalence","health_expenditure","gdp_per_capita",
            "vaccination_dtp3","water_access"]

print("=== EDA Summary ===")
for col in num_cols:
    d = df[col]
    print(f"{col:<25} mean={d.mean():.1f}  sd={d.std():.1f}  "
          f"skew={d.skew():.2f}  [{d.min():.0f}, {d.max():.0f}]")

print("\n=== Correlation with life_expectancy ===")
corr = df[num_cols].corr()["life_expectancy"].drop("life_expectancy").sort_values()
for var, r in corr.items():
    bar = "█" * int(abs(r) * 20)
    sign = "+" if r > 0 else "-"
    print(f"  {var:<25} r={r:+.3f}  {sign}{bar}")

In [ ]:
# EDA 2×4 histogram grid
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
axes = axes.flatten()
colors = plt.cm.tab10.colors

for i, col in enumerate(num_cols):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=15, color=colors[i], edgecolor="white", alpha=0.8)
    ax.axvline(data.mean(),   color="red",   ls="--", lw=1.5, label="Mean")
    ax.axvline(data.median(), color="green", ls="--", lw=1.5, label="Median")
    ax.set_title(col.replace("_"," ").title(), fontsize=9, fontweight="bold")
    if i == 0:
        ax.legend(fontsize=7)

fig.suptitle("Capstone Part 3 — Univariate Distributions",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("capstone_output/eda_distributions.png", bbox_inches="tight")
plt.show()

---
## Part 4 — Statistical Testing (20 pts)

In [ ]:
print("=" * 62)
print("CAPSTONE — STATISTICAL TESTS")
print("=" * 62)

# ── Test 1: One-sample t-test ─────────────────────────────────────────
t1, p1 = stats.ttest_1samp(df["life_expectancy"], popmean=65.0)
print(f"\nT1 — One-sample: Is mean LE = 65 years?")
print(f"     t={t1:.3f}, p={p1:.4f}, mean={df['life_expectancy'].mean():.2f}")
print(f"     → {'SIGNIFICANT' if p1 < 0.05 else 'NOT significant'}")

# ── Test 2: Two-sample t-test ─────────────────────────────────────────
north  = df[df["region"]=="North Africa"]["life_expectancy"]
others = df[df["region"]!="North Africa"]["life_expectancy"]
t2, p2 = stats.ttest_ind(north, others, equal_var=False)
d2 = (north.mean() - others.mean()) / np.sqrt((north.std()**2 + others.std()**2)/2)
print(f"\nT2 — Two-sample: North Africa vs Sub-Saharan LE")
print(f"     t={t2:.3f}, p={p2:.4f}, Cohen's d={d2:.3f}")
print(f"     → {'SIGNIFICANT' if p2 < 0.05 else 'NOT significant'}")

# ── Test 3: One-way ANOVA ─────────────────────────────────────────────
grps = [g["life_expectancy"].values for _, g in df.groupby("region")]
F, p3 = stats.f_oneway(*grps)
grand = df["life_expectancy"].mean()
ss_b  = sum(len(g) * (g["life_expectancy"].mean() - grand)**2
            for _, g in df.groupby("region"))
ss_t  = sum((df["life_expectancy"] - grand)**2)
eta2  = ss_b / ss_t
print(f"\nT3 — ANOVA: LE across 5 regions")
print(f"     F={F:.3f}, p={p3:.4f}, η²={eta2:.4f}")
print(f"     → {'SIGNIFICANT' if p3 < 0.05 else 'NOT significant'}")

# ── Test 4: Pearson correlation ───────────────────────────────────────
r4, p4 = stats.pearsonr(np.log(df["gdp_per_capita"]), df["life_expectancy"])
print(f"\nT4 — Pearson: log(GDP) vs LE")
print(f"     r={r4:.4f}, p={p4:.6f}")
print(f"     → {'SIGNIFICANT' if p4 < 0.05 else 'NOT significant'}")

# ── Save test results ─────────────────────────────────────────────────
test_results = pd.DataFrame([
    {"Test": "One-sample t",      "Statistic": round(t1,3), "p-value": round(p1,4), "Sig": p1<0.05},
    {"Test": "Two-sample t",      "Statistic": round(t2,3), "p-value": round(p2,4), "Sig": p2<0.05},
    {"Test": "One-way ANOVA",     "Statistic": round(F,3),  "p-value": round(p3,4), "Sig": p3<0.05},
    {"Test": "Pearson (log GDP)", "Statistic": round(r4,4), "p-value": round(p4,6), "Sig": p4<0.05},
])
test_results.to_csv("capstone_output/statistical_tests.csv", index=False)
print("\nTest results saved → capstone_output/statistical_tests.csv")

---
## Part 5 — Modelling (20 pts)

In [ ]:
# ── 5A: Multiple Linear Regression — Predict life_expectancy ──────────
reg_features = ["log_gdp", "vaccination_dtp3", "water_access", "health_expenditure"]
X_reg = df[reg_features]
y_reg = df["life_expectancy"]

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.25, random_state=42)

mlr = LinearRegression()
mlr.fit(X_tr, y_tr)
y_pred = mlr.predict(X_te)

r2_test  = r2_score(y_te, y_pred)
rmse     = np.sqrt(mean_squared_error(y_te, y_pred))
cv_r2    = cross_val_score(mlr, X_reg, y_reg, cv=5, scoring="r2").mean()

print("5A — Multiple Linear Regression")
print(f"  Equation: LE = {mlr.intercept_:.2f}")
for feat, coef in zip(reg_features, mlr.coef_):
    print(f"            + {coef:+.3f} × {feat}")
print(f"  Test R²   = {r2_test:.4f}")
print(f"  5-fold CV R² = {cv_r2:.4f}")
print(f"  RMSE      = {rmse:.3f} years")

In [ ]:
# ── 5B: Classification — Predict high_le ─────────────────────────────
clf_features = ["log_gdp", "vaccination_dtp3", "water_access",
                "health_expenditure", "log_infant_mort"]
df["log_infant_mort"] = np.log(df["infant_mortality"])

X_clf = df[clf_features]
y_clf = df["high_le"]
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=42, stratify=y_clf
)

classifiers = {
    "Random Forest" : RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree" : DecisionTreeClassifier(max_depth=4, random_state=42),
}

print("\n5B — Classification Results")
best_name, best_acc = None, 0
model_results = {}

for name, clf in classifiers.items():
    clf.fit(X_tr_c, y_tr_c)
    y_pred_c = clf.predict(X_te_c)
    acc = accuracy_score(y_te_c, y_pred_c)
    cv  = cross_val_score(clf, X_clf, y_clf, cv=5).mean()
    model_results[name] = {"test_acc": acc, "cv_acc": cv, "model": clf}
    print(f"  {name:<20} test={acc:.4f}  cv={cv:.4f}")
    if acc > best_acc:
        best_acc, best_name = acc, name

print(f"\n  Best model: {best_name} (acc={best_acc:.4f})")
print("\nBest model classification report:")
best_model = model_results[best_name]["model"]
print(classification_report(y_te_c, best_model.predict(X_te_c),
                             target_names=["Low LE","High LE"]))

In [ ]:
# ── 5C: k-Means Clustering ────────────────────────────────────────────
clust_features = ["log_gdp", "life_expectancy",
                  "log_infant_mort", "vaccination_dtp3", "water_access"]
X_cl = df[clust_features]
scaler_cl = StandardScaler()
X_cl_s = scaler_cl.fit_transform(X_cl)

# Silhouette to choose k
sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cl_s)
    sil_scores[k] = silhouette_score(X_cl_s, labels)

best_k = max(sil_scores, key=sil_scores.get)
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_cl_s)

print(f"\n5C — k-Means Clustering")
print(f"  Best k = {best_k}  (silhouette = {sil_scores[best_k]:.4f})")
print("\nCluster Profiles (mean values):")
profile = df.groupby("cluster")[clust_features + ["life_expectancy", "health_index"]].mean().round(1)
print(profile.to_string())

print("\nCluster × Region:")
print(pd.crosstab(df["cluster"], df["region"]))

---
## Part 6 — Visualisation Dashboard (15 pts)

In [ ]:
# ── Capstone Dashboard ────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor("#F0F4F8")
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel A: Health Index ranking (top 12) ────────────────────────────
ax_a = fig.add_subplot(gs[0:2, 0])
top12 = df.nlargest(12, "health_index")[["country","region","health_index"]]
colors_a = [PALETTE[r] for r in top12["region"]]
ax_a.barh(top12["country"][::-1], top12["health_index"][::-1],
          color=colors_a[::-1], edgecolor="white", alpha=0.9)
ax_a.set_xlabel("Health Index (0–100)")
ax_a.set_title("A — Top 12: Health Index", fontweight="bold")
ax_a.set_facecolor("white")

# ── Panel B: GDP vs LE scatter ────────────────────────────────────────
ax_b = fig.add_subplot(gs[0:2, 1:3])
for region, grp in df.groupby("region"):
    sizes = (grp["vaccination_dtp3"] / 99) * 300 + 30
    ax_b.scatter(grp["log_gdp"], grp["life_expectancy"],
                 color=PALETTE[region], s=sizes, alpha=0.8,
                 edgecolors="white", label=region)
m, b_coef = np.polyfit(df["log_gdp"], df["life_expectancy"], 1)
x_fit = np.linspace(df["log_gdp"].min(), df["log_gdp"].max(), 100)
ax_b.plot(x_fit, m*x_fit+b_coef, "k--", lw=1.5)

# Annotate 3 notable countries
for _, row in df[df["country"].isin(["Mauritius","South Sudan","Nigeria"])].iterrows():
    ax_b.annotate(row["country"],
                  xy=(row["log_gdp"], row["life_expectancy"]),
                  xytext=(5, 3), textcoords="offset points", fontsize=8)
ax_b.set_xlabel("Log GDP per Capita")
ax_b.set_ylabel("Life Expectancy (yrs)")
ax_b.set_title(f"B — GDP vs LE (bubble = vaccination rate)\n"
               f"Regression: LE = {b_coef:.1f} + {m:.2f}·log(GDP),  r={r4:.3f}",
               fontweight="bold")
ax_b.legend(fontsize=7, loc="lower right")
ax_b.set_facecolor("white")

# ── Panel C: Violin — LE by region ───────────────────────────────────
ax_c = fig.add_subplot(gs[0:2, 3])
r_order = df.groupby("region")["life_expectancy"].median().sort_values().index
sns.violinplot(data=df, x="region", y="life_expectancy",
               order=r_order, palette=PALETTE,
               inner="box", alpha=0.8, ax=ax_c, orient="v")
ax_c.tick_params(axis="x", rotation=25, labelsize=8)
ax_c.set_xlabel("")
ax_c.set_title("C — LE by Region", fontweight="bold")
ax_c.set_facecolor("white")

# ── Panel D: Correlation heatmap ──────────────────────────────────────
ax_d = fig.add_subplot(gs[2, 0:2])
corr_mat = df[num_cols].corr()
mask_upper = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask_upper, annot=True, fmt=".1f",
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar=False,
            ax=ax_d, annot_kws={"size": 7})
ax_d.set_title("D — Correlation Matrix", fontweight="bold")
ax_d.tick_params(labelsize=7, rotation=45)

# ── Panel E: k-Means cluster profile ─────────────────────────────────
ax_e = fig.add_subplot(gs[2, 2:4])
prof = df.groupby("cluster")[["life_expectancy","health_index",
                                "gdp_per_capita","vaccination_dtp3"]].mean()
prof_norm = (prof - prof.min()) / (prof.max() - prof.min())
sns.heatmap(prof_norm.T, annot=prof.T.round(0),
            cmap="YlOrRd", fmt=".0f", cbar=False,
            ax=ax_e, annot_kws={"size": 9},
            xticklabels=[f"C{i+1}" for i in range(best_k)])
ax_e.set_title(f"E — k-Means Cluster Profiles (k={best_k})",
               fontweight="bold")
ax_e.tick_params(labelsize=8)

# Legend patches
handles = [mpatches.Patch(color=c, label=r) for r, c in PALETTE.items()]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("CAPSTONE DASHBOARD — Africa Health Analysis",
             fontsize=16, fontweight="bold", y=1.01)

fig.savefig("capstone_output/capstone_dashboard.png",
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Dashboard saved → capstone_output/capstone_dashboard.png")

---
## Part 7 — Written Report & Recommendations (15 pts)

Produce a structured written report using the template below.

In [ ]:
# Auto-generate a report text file
report_lines = [
    "=" * 70,
    "CAPSTONE REPORT — Africa Health Analysis",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    "=" * 70,
    "",
    "1. RESEARCH QUESTION",
    "   Which African countries have the best health outcomes, what",
    "   economic and infrastructure factors drive them, and can we",
    "   predict and classify countries accurately?",
    "",
    "2. DATASET",
    f"   54 African countries × 10 indicators (2022 synthetic data)",
    f"   Clean dataset shape: {df.shape}",
    "",
    "3. DATA QUALITY",
    f"   Missing values imputed: region-conditional median strategy",
    f"   Duplicates removed: 2",
    "",
    "4. KEY EDA FINDINGS",
    f"   Life expectancy: mean={df['life_expectancy'].mean():.1f}, sd={df['life_expectancy'].std():.1f}",
    f"   Strongest negative correlation: infant_mortality (r={df['life_expectancy'].corr(df['infant_mortality']):.3f})",
    f"   Strongest positive correlation: log(GDP) (r={r4:.3f})",
    "",
    "5. STATISTICAL RESULTS",
    f"   One-sample t-test (μ=65): t={t1:.3f}, p={p1:.4f}",
    f"   North vs Sub-Saharan LE: t={t2:.3f}, p={p2:.4f}, d={d2:.3f}",
    f"   ANOVA (5 regions): F={F:.3f}, p={p3:.4f}, η²={eta2:.4f}",
    "",
    "6. MODEL PERFORMANCE",
    f"   MLR (LE prediction): Test R²={r2_test:.4f}, RMSE={rmse:.3f} yrs",
    f"   Best classifier ({best_name}): Test Acc={best_acc:.4f}",
    f"   k-Means (k={best_k}): Silhouette={sil_scores[best_k]:.4f}",
    "",
    "7. RECOMMENDATIONS",
    "   1. Prioritise water infrastructure in West and Central Africa",
    "      (water_access has second-strongest correlation with LE).",
    "   2. Increase vaccination coverage — DTP3 strongly predicts LE",
    "      beyond GDP level alone.",
    "   3. Cluster 1 (high burden) countries need targeted health",
    "      expenditure allocation to close the regional gap.",
    "   4. Adopt the Health Index as a monitoring KPI — it integrates",
    "      four indicators into a single comparable score.",
    "",
    "8. LIMITATIONS",
    "   • Synthetic data — real-world data may show stronger confounding",
    "   • Cross-sectional snapshot — no causal inference possible",
    "   • Small N (54) limits statistical power for complex models",
    "   • Health index weights are assumed, not empirically derived",
    "=" * 70,
]

report_text = "\n".join(report_lines)
print(report_text)

with open("capstone_output/capstone_report.txt", "w") as f:
    f.write(report_text)
print("\nReport saved → capstone_output/capstone_report.txt")

In [ ]:
# Save a JSON provenance log
provenance = {
    "project"          : "Africa Health Capstone",
    "date"             : datetime.now().isoformat(),
    "dataset"          : {"rows": int(df.shape[0]), "cols": int(df.shape[1])},
    "cleaning"         : {"duplicates_removed": 2, "imputation": "region-conditional median"},
    "features_created" : ["log_gdp","log_infant_mort","log_maternal_mort",
                          "health_index","disease_burden","high_le","gdp_band"],
    "models"           : {
        "MLR"            : {"test_r2": round(r2_test, 4), "rmse": round(rmse, 4)},
        best_name        : {"test_acc": round(best_acc, 4)},
        "KMeans"         : {"k": best_k, "silhouette": round(sil_scores[best_k], 4)},
    },
    "outputs"          : [
        "raw_data.csv", "clean_data.csv", "statistical_tests.csv",
        "capstone_dashboard.png", "capstone_report.txt", "provenance.json"
    ]
}

with open("capstone_output/provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

print("Provenance log saved → capstone_output/provenance.json")
print("\nAll capstone outputs:")
for fname in sorted(os.listdir("capstone_output")):
    size = os.path.getsize(f"capstone_output/{fname}")
    print(f"  {fname:<40} {size:>8,} bytes")

---
## Your Capstone Tasks

The cells above provide a **complete reference implementation**.  
For your assessed submission, you must extend it with the following:

### Task 1 — Extended EDA (Part 3 extension) (20 pts)
Add to the EDA section:
- A pair plot for 5 variables coloured by region
- A regional heatmap showing mean values of all indicators (normalised)
- A written paragraph (150 words) identifying 3 patterns

### Task 2 — Additional Model (Part 5 extension) (25 pts)
Build a **Logistic Regression** with Ridge regularisation to predict `high_le`.  
Report: accuracy, F1, ROC curve, odds ratios. Compare to Random Forest.

### Task 3 — Custom Dashboard (Part 6 extension) (25 pts)
Design your own `3 × 3` GridSpec dashboard focused on a question of your choice  
(e.g., "Where should foreign health aid be targeted?").  
Include at least 1 annotated storytelling chart.

### Task 4 — Executive Summary (Part 7 extension) (30 pts)
Write an executive summary (400–500 words) covering:
1. Research question and data overview
2. Three key statistical findings (with test statistics)
3. Model performance summary
4. Three actionable policy recommendations backed by data
5. Two methodological limitations

In [ ]:
# Task 1 — Extended EDA
pass  # TODO

In [ ]:
# Task 2 — Logistic Regression with Regularisation
pass  # TODO

In [ ]:
# Task 3 — Custom Dashboard
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 3, figure=fig)
# TODO: Fill 9 panels

### Task 4 — Executive Summary (400–500 words)

> *Type your executive summary here.*
>
> **1. Research Question & Data:**
>
> **2. Key Findings:**
>
> **3. Model Performance:**
>
> **4. Recommendations:**
>
> **5. Limitations:**

---
## Grading Rubric

| Task | Points | Criteria |
|---|---|---|
| **Task 1: Extended EDA** | 20 | Pair plot ✓, regional heatmap ✓, 3 patterns identified ✓ |
| **Task 2: Logistic Regression** | 25 | Model runs ✓, F1/ROC ✓, comparison to RF ✓, interpretation ✓ |
| **Task 3: Dashboard** | 25 | 9 panels ✓, clear story ✓, annotated chart ✓, clean design ✓ |
| **Task 4: Written Report** | 30 | Structure ✓, accuracy ✓, stats cited ✓, recommendations ✓, limitations ✓ |
| **Total** | **100** | |

### Grade Boundaries
| Range | Grade |
|---|---|
| 90–100 | A+ |
| 80–89 | A |
| 70–79 | B |
| 60–69 | C |
| < 60 | Needs improvement |

---
## Course Summary — 10 Weeks

| Week | Topic | Key Skill Gained |
|---|---|---|
| 1 | Introduction | Data literacy, workflow map |
| 2 | Data Collection | Multi-source ingestion, data quality |
| 3 | Data Cleaning | Imputation, outlier handling, normalisation |
| 4 | EDA | Descriptive stats, distribution analysis, correlation |
| 5 | Visualisation | Chart design, Matplotlib/Seaborn mastery, dashboards |
| 6 | Statistics | Hypothesis testing, CIs, effect size |
| 7 | Regression | Linear/logistic models, diagnostics, prediction |
| 8 | Classification | Decision trees, kNN, k-means, evaluation metrics |
| 9 | Advanced | Feature engineering, PCA, SQL, regularisation |
| 10 | Capstone | Full end-to-end professional analysis pipeline |

---
🎓 **Congratulations on completing the Data Analysis course!**  
You can now confidently load, clean, explore, model, and communicate insights from real-world data.